In [1]:
import torch
from torchsummary import summary
import numpy as np
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchvision.models import vgg16_bn, VGG16_BN_Weights


from torchvision import transforms

from cvat_sdk import make_client
from cvat_sdk.pytorch import ProjectVisionDataset

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <9A4710B9-0DA3-36BB-9129-645F282E64B2> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so
  Expected in:     <ECC148AF-20FF-3EEE-BC75-4DD3E7455393> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
IMAGE_SHAPE = (640, 640)
class RLETransform(torch.nn.Module):
    '''
    transforms CVAT data to proper objects of labels, bboxes and masks
    '''
    def forward(self, Target):
        labels = []
        bboxes = []
        masks = []
        for data in Target.annotations.shapes:
            labels.append(data['label_id'])
            bboxes.append(data['points'][-4:])
            masks.append(self.rle_to_mask(data['points'][:-4], bboxes[-1]))

        return labels, bboxes, masks

    
    def rle_to_mask(self, rle, bbox):
        width  = int(bbox[2]-bbox[0])+1
        height = int(bbox[3]-bbox[1])+1
        top = int(bbox[1])
        left = int(bbox[0])
        
        mask = np.zeros(width * height, dtype=np.uint8)
        
        rle.insert(0,0.0)
        rle_pairs = np.array(rle, dtype=np.int16)
        pos = 0
        for i in range(0, len(rle_pairs), 2):
            start = pos
            end = pos + rle_pairs[i]
            mask[start:end] = 1  # Устанавливаем пиксели в белый цвет (255)
            pos = end + rle_pairs[i + 1]  # Пропускаем следующие пиксели

        full_mask = np.zeros(IMAGE_SHAPE)
        full_mask[top:top+height, left:left+width] = mask.reshape((height, width))
        
        
        return full_mask

## создаем датасет из загруженных с CVAT данных

In [ ]:
class_dict = {
    9: 'Apple',
    10: 'Hurma',
    11: 'Orange',
    12: 'Pear',
    13: 'Kiwi',
    14: 'Tangerine'
}

# log into the CVAT server
with make_client(host="http://10.162.1.50:8080", credentials=('admin', 'qaedwsrf123')) as client:
    # get the dataset comprising all tasks for the Validation subset of project 12345
    dataset = ProjectVisionDataset(client, project_id=2,
                                  transform = transforms.ToTensor(),
                                  target_transform = RLETransform())

    print(len(dataset))

In [2]:
class Spine(nn.Module):
    def __init__(self, vgg):
        super(Spine, self).__init__()
        
        for i, child in enumerate(vgg.children()):
            for param in child.parameters():
                param.requires_grad = False
                
            if i == 0:
                layers_list = child
        
        self.conv = nn.Sequential(*layers_list[:-1])

    def forward(self, x):

        x = self.conv(x)

        return x
    

In [3]:
DETECTION_CLASSES = 6
# !!!! класс с индексом 0 - задний фон !!!!
class СlassifierHead(nn.Module):
    def __init__(self):
        super(СlassifierHead, self).__init__()

        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(in_features = 8192, out_features = 2048, bias=True),
            nn.BatchNorm1d(2048, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 2048, out_features = 2048, bias=True),
            nn.BatchNorm1d(2048, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 2048, out_features = 1024, bias=True),
            nn.BatchNorm1d(1024, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 1024, out_features = 512, bias=True),
            nn.BatchNorm1d(512, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 512, out_features = DETECTION_CLASSES + 1, bias=True),
            nn.Softmax(dim=0)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc(x)

        return x

In [ ]:
def get_GT_of_bbox(bbox, GT_bboxes, GT_labels, overlap=0.4):
    '''
    computing ground truth labels of given bbox in original image cordinates
    '''
    x1, y1, x2, y2 = bbox
    bbox_area = (x2 - x1) * (y2 - y1)
    if bbox_area == 0:
        return 0
    
    max_overlap = 0
    best_id = 0
    best_bbox = None
    
    for gt_bbox, label in zip(GT_bboxes, GT_labels):
        gt_x1, gt_y1, gt_x2, gt_y2 = gt_bbox
        
        # Вычисляем координаты пересечения
        inter_x1 = max(x1, gt_x1)
        inter_y1 = max(y1, gt_y1)
        inter_x2 = min(x2, gt_x2)
        inter_y2 = min(y2, gt_y2)
        
        # Вычисляем ширину и высоту пересечения
        inter_width = inter_x2 - inter_x1
        inter_height = inter_y2 - inter_y1
        
        # Если пересечение существует
        if inter_width > 0 and inter_height > 0:
            inter_area = inter_width * inter_height
            overlap_ratio = inter_area / bbox_area
            
            # Проверяем условие на минимальное пересечение и сравниваем с текущим максимальным пересечением
            if overlap_ratio >= overlap and inter_area > max_overlap:
                max_overlap = inter_area
                best_id = label
                best_bbox = gt_bbox
    
    return best_id, best_bbox

# класс всей mask r-cnn целиком
class Segment:
    def __init__(
        self,
        spine,
        classifier,
        classifier_optim,
        classifier_criterion,
        
        regressor = None,
        segmentator = None
    ):
        self.spine = spine
        self.spine.eval()
        
        self.classifier = classifier
        self.classifier_optim = classifier_optim
        self.classifier_criterion = classifier_criterion
        self.classifier_losses = []
        
        self.regressor = regressor
        self.segmentator = segmentator

    # лучше это переоформить как связку хребта и классификатора
    def spine_to_classifier(self, batch, train=True, targets=None):
        # getting image features
        with torch.no_grad():
            sample = self.spine(batch)
        
        boxes = [torch.unsqueeze(el, dim=0) for el in self.anchor_boxes()]
        boxes = torch.cat(boxes, dim=0)

        # getting classifier labels for each achor_box of single image in batch
        BATCH = 0
            
        ancors_batch = sample[BATCH, :, boxes[0,0]:boxes[0,2], boxes[0,1]:boxes[0,3]]
        ancors_batch = torch.unsqueeze(ancors_batch, dim=0)
        
        for i in range(1, len(boxes)):
            batch_element = sample[BATCH, :, boxes[i,0]:boxes[i,2], boxes[i,1]:boxes[i,3]]
            batch_element = torch.unsqueeze(batch_element, dim=0)
            
            ancors_batch = torch.cat((ancors_batch, batch_element), dim=0)


        # отправляем полученный батч в классификатор
        if train:
            self.classifier.train()
            self.classifier_optim.zero_grad()

            classifierHead_response = self.classifier(ancors_batch.detach())

            loss = self.classifier_criterion(classifierHead_response, targets)
            loss.backward()
            self.classifier_optim.step()
            
            self.classifier_losses.append(loss.item())
        else:
            self.classifier.eval()
            with torch.no_grad():
                classifierHead_response = self.classifier(ancors_batch.detach())
        
        
        # от classifierHead_response ищем лосс и делаем step при обучении
        # сделать метод для сопоставления места ancor_box и места bbox из GT

        
        verdict = torch.argmax(classifierHead_response, dim=1)


        # значения всех боксов которые не фон
        boxes_features = ancors_batch[verdict!=0].detach()
        # позиции всех боксов которые не фон
        positions = boxes[verdict!=0]
        # классы этих боксов
        class_ids = verdict[verdict!=0]

        # отправляем полученный батч регрессору
        return boxes_features, positions, class_ids





        

    def foreground_to_regressor(self, boxes_features, positions, train=True, targets=None):
        pass

    # здесь будем определять GT для всех голов и вызывать их в нужном режиме
    def train_step(self, image, targets):
        labels, bboxes, masks = targets

        # собираем истинные метки классов всех ancor_box и, если они не фон, их истинные размеры
        ancor_boxes_GT_labels = []
        ancor_boxes_GT = []
        
        
        boxes_features, positions, class_ids = self.spine_to_classifier(image, targets=

    # вызываем train_step для всех элементов датасета
    def train(self, dataset):
        pass

    # вызываем все головы в рабочем режиме
    def __call__(self, image):
        pass

        
    def image_bbox_cords(self, bbox, scale_factor=16):
        bbox = bbox*scale_factor
        return bbox

    def sample_bbox_cords(self, bbox, scale_factor=0.0625):
        bbox = bbox*scale_factor
        return bbox


    def anchor_boxes(self, sample_size = (40,40), box_size = (4,4), stride = 4):
        '''
        returns all boxes by parameters y_top, x_left, y_bot, x_right
        box tensor will be: sample[BATCH, :, box[0]:box[2], box[1]:box[3]]
        '''
        for y in range(0, sample_size[0], stride):
            for x in range(0, sample_size[1], stride):
                yield torch.tensor([y,x,y+box_size[0],x+box_size[1]])

In [4]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(device)

model = Spine(vgg16_bn(weights = VGG16_BN_Weights.IMAGENET1K_V1)).to(device)
classifier = СlassifierHead()

cpu
